In [1]:
import numpy as np

from framework.Tensor import Tensor
from framework.SGD import SGD
from framework.Sequential import Sequential
from framework.Linear import Linear
from framework.Tanh import Tanh
from framework.Sigmoid import Sigmoid
from framework.MSELoss import MSELoss

In [2]:
x = Tensor([1, 2, 3])
y = Tensor([4, 5, 6])
z = x + y
z.dump()

Tensor ID: 802
Data: [5 7 9]
Autograd: False
Gradient: None
Creation Operation: None

Parents:
  None

Children:
  No children


In [3]:
z.backward(Tensor(np.array([5, 6, 7])))
z.dump()

Tensor ID: 802
Data: [5 7 9]
Autograd: False
Gradient: None
Creation Operation: None

Parents:
  None

Children:
  No children


In [4]:
a = Tensor([1, 2, 3, 4, 5])
b = Tensor([2, 2, 2, 2, 2])
c = Tensor([5, 4, 3, 2, 1])
d = Tensor([-1, -2, -3, -4, -5])
e = a + b
f = c + d
g = e + f
g.backward(Tensor(np.array([1, 1, 1, 1, 1])))
b.dump()

Tensor ID: 729
Data: [2 2 2 2 2]
Autograd: False
Gradient: None
Creation Operation: None

Parents:
  None

Children:
  No children


In [5]:
# Текущая версия класса Tensor поддерживает обратное распространение градиентов только один раз
# для одной переменной, но иногда во время прямого прохода мы будем использовать тот же тензор
# несколько раз, и поэтому несколько частей графа будут распространять
# градиенты обратно в тот же Тензор.

a = Tensor([1, 2, 3, 4, 5], id='a')
b = Tensor([2, 2, 2, 2, 2], id='b')
c = Tensor([5, 4, 3, 2, 1], id='c')

d = a + b
e = b + e
f = d + e

f.backward(Tensor([1, 1, 1, 1, 1]))
f.dump()

Tensor ID: 830
Data: [ 8 10 12 14 16]
Autograd: False
Gradient: None
Creation Operation: None

Parents:
  None

Children:
  No children


# Design basic Tensor class

In [6]:
# Проверяем, работает ли автоград

a = Tensor([1, 2, 3, 4, 5], id="a", autograd=True)
b = Tensor([2, 2, 2, 2, 2], id="b", autograd=True)
c = Tensor([5, 4, 3, 2, 1], id="c", autograd=True)

d = a + b
e = b + c
f = d + e

f.backward(Tensor([1, 1, 5, 1, 1]))

print(b.grad.data == [2, 2, 10, 2, 2])
b.dump()

[ True  True  True  True  True]
Tensor ID: b
Data: [2 2 2 2 2]
Autograd: True
Gradient: Tensor(935) Data: [ 2  2 10  2  2]
Creation Operation: None

Parents:
  None

Children:
  Child ID: 901, Count: 0
  Child ID: 723, Count: 0


In [7]:
# test __add__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])
a + b

Tensor(904) Data: [5 7 9]

In [8]:
# test __neg__

a = Tensor([1, 2, 3])
-a

Tensor(844) Data: [-1 -2 -3]

In [9]:
# test __sub__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])
(a - b)

Tensor(60) Data: [-3 -3 -3]

In [10]:
# test __mul__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])

(a * b)

Tensor(598) Data: [ 4 10 18]

In [11]:
# test __matmul__

a = Tensor([1, 2, 3])
b = Tensor([4, 5, 6])

expected_value = np.array([1, 2, 3]) @ np.array([4, 5, 6])
print(f"Expected: {expected_value}")

print(a @ b)
print(a.mm(b))

Expected: 32
Tensor(356) Data: 32
Tensor(97) Data: 32


In [12]:
# test transpose

a = Tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
a.transpose()

Tensor(29) Data: [[1 4 7]
 [2 5 8]
 [3 6 9]]

In [13]:
# test sum

a = Tensor([1, 2, 3])
print(a.sum(0))
print()

b = Tensor([[1, 2, 3], [4, 5, 6]])
print(b.sum(0))
print(b.sum(1))
print()

c = Tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(c.sum(0))
print(c.sum(1))
c.data

Tensor(354) Data: 6

Tensor(326) Data: [5 7 9]
Tensor(304) Data: [ 6 15]

Tensor(421) Data: [12 15 18]
Tensor(65) Data: [ 6 15 24]


array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9]])

In [14]:
# test sum

a = Tensor([1, 2, 3])
print(a.expand(0, 2))

b = Tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(b.expand(0, 2))
print(b.expand(1, 2))
print(b.expand(2, 2))

Tensor(590) Data: [[1 2 3]
 [1 2 3]]
Tensor(999) Data: [[[1 2 3]
  [4 5 6]
  [7 8 9]]

 [[1 2 3]
  [4 5 6]
  [7 8 9]]]
Tensor(580) Data: [[[1 2 3]
  [1 2 3]]

 [[4 5 6]
  [4 5 6]]

 [[7 8 9]
  [7 8 9]]]
Tensor(472) Data: [[[1 1]
  [2 2]
  [3 3]]

 [[4 4]
  [5 5]
  [6 6]]

 [[7 7]
  [8 8]
  [9 9]]]


# Using Tensor class to train NN

In [15]:
import numpy as np

np.random.seed(0)

dataset = np.array([
    [0, 0, 0],
    [0, 1, 1],
    [1, 0, 0],
    [1, 1, 1],
])

X_train = dataset[:, :2]  # (4, 2)
Y_train = dataset[:, 2:]  # (4, 1)

hidden_size = 3
epochs = 10
alpha = 0.1

weights_0_1 = np.random.rand(X_train.shape[1], hidden_size)  # (2, 3)
weights_1_2 = np.random.rand(hidden_size, Y_train.shape[1])  # (3, 1)

for i in range(epochs):
    # Прямой проход
    layer_1 = X_train @ weights_0_1  # (4, 3)
    layer_2 = layer_1 @ weights_1_2  # (4, 1)

    # На сколько мимо от цели
    diff = layer_2 - Y_train  # (4, 1)

    # Квадрат убирает знак и усиливает ошибку
    # И суммируем ее чтобы использовать как метрику
    loss = (diff ** 2).sum(0)  # (1,)

    # Обратное распространение
    # 1. Считаем градиенты между выходом и внутренем слоем
    layer_1_grad = diff @ weights_1_2.T  # (4, 1) @ (1, 10) = (4, 10)
    # 2. Считаем на сколько нужно изменить вес между 1 и 2
    weights_1_2_delta = layer_1.T @ diff  # (3, 4) @ (4, 1) = (3, 1)
    # 3. Считаем на сколько нужно изменить вес между 0 и 1
    weights_0_1_delta = X_train.T @ layer_1_grad  # (3, 4) @ (4, 3) = (3, 3)

    # Обновляем веса с учетом шага обучения
    weights_1_2 -= weights_1_2_delta * alpha
    weights_0_1 -= weights_0_1_delta * alpha

    print(loss[0])


5.066439994622395
0.4959907791902342
0.4180671892167177
0.35298133007809646
0.2972549636567377
0.2492326038163328
0.20785392075862477
0.17231260916265176
0.14193744536652986
0.11613979792168384


In [16]:
import numpy as np

np.random.seed(42)

dataset = np.array([
    [0, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [1, 1, 1, 1],
])

X_train = dataset[:, :3]  # (8, 3)
Y_train = dataset[:, 3:]  # (8, 1)

hidden_size = 10
epochs = 50
alpha = 0.01

weights_0_1 = np.random.rand(X_train.shape[1], hidden_size)  # (3, 10)
weights_1_2 = np.random.rand(hidden_size, Y_train.shape[1])  # (10, 1)

for i in range(epochs):
    # Прямой проход
    layer_1 = X_train @ weights_0_1  # (8, 10)
    layer_2 = layer_1 @ weights_1_2  # (8, 1)

    # На сколько мимо от цели
    diff = layer_2 - Y_train  # (8, 1)

    # Квадрат убирает знак и усиливает ошибку
    # И суммируем ее чтобы использовать как метрику
    loss = (diff ** 2).sum(0)

    # Обратное распространение
    # 1. Считаем градиенты между выходом и внутренем слоем
    layer_1_grad = diff @ weights_1_2.T  # (8, 1) @ (1, 10) = (8, 10)
    # 2. Считаем на сколько нужно изменить вес между 1 и 2
    weights_1_2_delta = layer_1.T @ diff  # (10, 8) @ (8, 1) = (10, 1)
    # 3. Считаем на сколько нужно изменить вес между 0 и 1
    weights_0_1_delta = X_train.T @ layer_1_grad  # (3, 8) @ (8, 10) = (3, 10)

    # Обновляем веса с учетом шага обучения
    weights_1_2 -= weights_1_2_delta * alpha
    weights_0_1 -= weights_0_1_delta * alpha

    if (i % 5 == 0 or i == epochs - 1):
        print(str(i) + ": " + str(round(loss[0], 4)))


0: 60.259
5: 1.5034
10: 1.1035
15: 0.894
20: 0.7789
25: 0.7141
30: 0.677
35: 0.6556
40: 0.643
45: 0.6357
49: 0.632


In [17]:
import numpy as np

np.random.seed(42)

dataset = np.array([
    [0, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [1, 1, 1, 1],
])

X_train = Tensor(dataset[:, :3], autograd=True)  # (8, 3)
Y_train = Tensor(dataset[:, 3:], autograd=True)  # (8, 1)

hidden_size = 10
epochs = 10
alpha = 0.01

weights = list()
weights.append(Tensor(np.random.rand(3, hidden_size), autograd=True))
weights.append(Tensor(np.random.rand(hidden_size, 1), autograd=True))

for i in range(epochs):

    predict = X_train @ weights[0] @ weights[1]
    loss = ((predict - Y_train) * (predict - Y_train)).sum(0)

    loss.backward(Tensor(np.ones_like(loss.data)))

    for weight in weights:
        weight.data -= weight.grad.data * alpha
        weight.grad.data *= 0

    print(str(i) + ": " + str(round(loss.data[0], 4)))

0: 46.2668
1: 4.7614
2: 0.7045
3: 0.6878
4: 0.6797
5: 0.6727
6: 0.6666
7: 0.6614
8: 0.6568
9: 0.6528


# Auto optimization

In [18]:
import numpy as np

np.random.seed(0)

dataset = np.array([
    [0, 0, 0],
    [0, 1, 1],
    [1, 0, 0],
    [1, 1, 1],
])

X_train = dataset[:, :2]  # (4, 2)
Y_train = dataset[:, 2:]  # (4, 1)

data = Tensor(X_train, autograd=True)
target = Tensor(Y_train, autograd=True)

w = list()
w.append(Tensor(np.random.rand(2, 3), autograd=True))
w.append(Tensor(np.random.rand(3, 1), autograd=True))

optim = SGD(params=w, alpha=0.1)

for i in range(10):
    pred = data @ w[0] @ w[1]
    loss = ((pred - target) * (pred - target)).sum(0)
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()

    print(loss.data[0])


0.5812830360381691
0.4898814918030678
0.4137511099699248
0.34489412208720704
0.2821012414527573
0.2254484048015707
0.17538852854551776
0.13242309965665466
0.09682768724516845
0.0684936057498309


In [19]:
np.random.seed(0)

data = Tensor(np.array([[0, 0], [0, 1], [1, 0], [1, 1]]), autograd=True)
target = Tensor(np.array([[0], [1], [0], [1]]), autograd=True)
model = Sequential([
    Linear(2, 3),
    Tanh(),
    Linear(3, 1),
    Sigmoid()
])

optim = SGD(params=model.get_params(), alpha=0.1)
criterion = MSELoss()

for i in range(10):
    pred = model.forward(data)
    loss = criterion.forward(pred, target)

    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()

    print(loss.data[0])

0.9178537085622939
0.906026923428229
0.895107247612061
0.8850568330613013
0.8758296838069995
0.867373899547553
0.8596338157893205
0.8525519411676739
0.8460706267807379
0.8401334348482176
